In [4]:
# ==============================================================================
# OLYMPUS TECH 4.0 - BONUS ROUND: REBUILDING EDEN (INFO_REAPER ML PIPELINE)
# ==============================================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

print("--- Step 1: Loading Cleaned Dataset & Test Dataset ---")
# Load your verified cleaned training dataset
df_train = pd.read_csv('InfoReaper_cleaned_dataset.csv')

# Load the unseen test dataset provided for the challenge
# (Ensure your test file name matches what you have in Colab/GitHub)
df_test = pd.read_csv('test.csv')

print(f"Training Data Shape: {df_train.shape}")
print(f"Test Data Shape: {df_test.shape}")

# ==============================================================================
# STEP 2: FEATURE ENGINEERING & TARGET CREATION
# ==============================================================================
print("\n--- Step 2: Defining Target & Preparing Features ---")

# Rule: Planets with a settlement index of 60 or higher are considered viable (True)
df_train['viable_planet'] = (df_train['settlement_index'] >= 60)

print(f"Target Distribution (Viable vs Unviable):\n{df_train['viable_planet'].value_counts(normalize=True) * 100}%")

# Identify and drop target, IDs, and raw text names from training features
drop_cols = ['planetid', 'planetId', 'planet_name', 'settlement_index', 'viable_planet']
X_train_raw = df_train.drop(columns=[col for col in drop_cols if col in df_train.columns])

# Save test planet IDs for final submission mapping
test_id_col = 'planetId' if 'planetId' in df_test.columns else ('planetid' if 'planetid' in df_test.columns else df_test.columns[0])
test_planet_ids = df_test[test_id_col]

X_test_raw = df_test.drop(columns=[col for col in ['planetid', 'planetId', 'planet_name', 'settlement_index'] if col in df_test.columns])

# ==============================================================================
# STEP 3: HANDLE CATEGORICAL VARIABLES & COLUMN ALIGNMENT
# ==============================================================================
print("\n--- Step 3: One-Hot Encoding and Column Alignment ---")

# Combine temporarily to ensure identical dummy columns across train and test sets
combined = pd.concat([X_train_raw, X_test_raw], axis=0, ignore_index=True)
combined_encoded = pd.get_dummies(combined, drop_first=True)

# Split back into train and test sets
X_train = combined_encoded.iloc[:len(X_train_raw)].fillna(0)
X_test = combined_encoded.iloc[len(X_train_raw):].fillna(0)

y_train = df_train['viable_planet']

print(f"Processed Training Features Shape: {X_train.shape}")
print(f"Processed Test Features Shape: {X_test.shape}")

# ==============================================================================
# STEP 4: MODEL TRAINING & LOCAL VALIDATION
# ==============================================================================
print("\n--- Step 4: Training & Evaluating Random Forest Classifier ---")

# Train-test split on training data to evaluate locally without data leakage
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

# Initialize robust Random Forest model
rf_model = RandomForestClassifier(n_estimators=150, max_depth=12, random_state=42, class_weight='balanced')
rf_model.fit(X_tr, y_tr)

# Local Validation Metrics
y_val_pred = rf_model.predict(X_val)
print(f"Local Validation Accuracy: {accuracy_score(y_val, y_val_pred):.4f}")
print("\nClassification Report:\n", classification_report(y_val, y_val_pred))

# ==============================================================================
# STEP 5: FINAL RETRAINING & GENERATING SUBMISSION CSV
# ==============================================================================
print("\n--- Step 5: Retraining on Full Dataset & Predicting Unseen Test Data ---")

# Retrain on 100% of the training data for maximum predictive power
rf_model.fit(X_train, y_train)

# Predict on the unseen test dataset (Output strictly boolean: True / False)
final_predictions = rf_model.predict(X_test)

# Construct final submission DataFrame matching requested schema
submission_df = pd.DataFrame({
    'planetId': test_planet_ids,
    'viable_planet': final_predictions
})

# Export to CSV
output_filename = 'InfoReaper_predicted.csv'
submission_df.to_csv(output_filename, index=False)

print(f"\nSuccess! Predictions saved successfully to '{output_filename}'.")
print(f"Total rows predicted: {len(submission_df)}")
print("\nPreview of final submission file:")
print(submission_df.head(10))

--- Step 1: Loading Cleaned Dataset & Test Dataset ---
Training Data Shape: (6395, 31)
Test Data Shape: (1600, 30)

--- Step 2: Defining Target & Preparing Features ---
Target Distribution (Viable vs Unviable):
viable_planet
False    57.654418
True     42.345582
Name: proportion, dtype: float64%

--- Step 3: One-Hot Encoding and Column Alignment ---
Processed Training Features Shape: (6395, 788)
Processed Test Features Shape: (1600, 788)

--- Step 4: Training & Evaluating Random Forest Classifier ---
Local Validation Accuracy: 0.8124

Classification Report:
               precision    recall  f1-score   support

       False       0.79      0.91      0.85       737
        True       0.85      0.68      0.75       542

    accuracy                           0.81      1279
   macro avg       0.82      0.79      0.80      1279
weighted avg       0.82      0.81      0.81      1279


--- Step 5: Retraining on Full Dataset & Predicting Unseen Test Data ---

Success! Predictions saved succes